In [1]:
import os
from tqdm import tqdm
from pdf2image import convert_from_path
import pytesseract
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_ollama.llms import OllamaLLM

# Global variable to store the QA chain
qa_chain = None

os.environ["TOKENIZERS_PARALLELISM"] = "false"

def setup_qa_chain(pdf_directory, persist_directory="./chroma_db_policies", base_url="http://10.50.10.240:10023/"):
    global qa_chain  # declare that we're modifying the global variable
    documents = []

    # Process each PDF file with a progress bar
    for filename in tqdm(os.listdir(pdf_directory), desc="Processing PDFs"):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(pdf_directory, filename)
            try:
                pages = convert_from_path(pdf_path)
            except Exception as e:
                print(f"Error converting {filename}: {e}")
                continue

            for page_num, page in enumerate(pages, start=1):
                try:
                    text = pytesseract.image_to_string(page)
                    metadata = {"source": filename, "page": page_num}
                    documents.append(Document(page_content=text, metadata=metadata))
                except Exception as e:
                    print(f"Error processing page {page_num} of {filename}: {e}")

    # Chunk documents with a progress bar
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunked_documents = []
    for doc in tqdm(documents, desc="Chunking documents"):
        chunks = text_splitter.split_text(doc.page_content)
        for chunk in chunks:
            chunked_documents.append(Document(page_content=chunk, metadata=doc.metadata))

    # Create embeddings from the chunked documents
    embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(chunked_documents, embedding_model, persist_directory=persist_directory)
    vectorstore.persist()

    print("Embeddings generated and stored in Chroma DB using all‑MiniLM‑L6‑v2 HuggingFace Embeddings.")

    # Instantiate the Ollama LLM
    llm = OllamaLLM(model="llama3.2", base_url=base_url)

    # Build the RetrievalQA chain and assign it to the global variable
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    )

    print("RetrievalQA chain is set up and ready for queries using Ollama LLM.")

    print("This is the value of qa_chain from setup:", qa_chain)

setup_qa_chain('./policies')

def get_response(question):
    print("This is the value of qa_chain from response:", qa_chain)
    if qa_chain is None:
        return "QA chain is not set up yet. Please initialize it first."
    answer = qa_chain.run(question)
    return answer

Chunking documents: 100%|██████████| 92/92 [00:00<00:00, 61094.99it/s]
/var/folders/wp/nyzlhv2n30n3bs8b5qvkgvg00000gn/T/ipykernel_32136/310785198.py:48: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Embeddings generated and stored in Chroma DB using all‑MiniLM‑L6‑v2 HuggingFace Embeddings.
RetrievalQA chain is set up and ready for queries using Ollama LLM.
This is the value of qa_chain from setup: verbose=False combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:"), llm=OllamaLLM(model='llama3.2', base_url='http://10.50.10.240:10023/'), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_variable_name='context') retriever=VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], v

/var/folders/wp/nyzlhv2n30n3bs8b5qvkgvg00000gn/T/ipykernel_32136/310785198.py:50: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [4]:
import pandas as pd
import evaluate
from bert_score import score
from IPython.display import display

# --- Define Test Questions and Ground Truths ---
# Each question asks for the definition of a specific term from the CPCS policy,
# and the ground_truths list contains the corresponding "correct" definition.
test_questions = [
    # "What is the definition of 'Client' according to CPCS Anti-Corruption Compliance Program?",
    # "What is the definition of 'CPCS Global Anti-Corruption Training' according to CPCS Anti-Corruption Compliance Program??",
    # "What is the definition of 'Partners' according to CPCS Anti-Corruption Compliance Program??",
    "What is the definition of 'Staff' according to CPCS Anti-Corruption Compliance Program??",
    "What is the definition of 'Due Diligence' according to CPCS Anti-Corruption Compliance Program??",
    "What is the definition of 'Whistleblower' according to CPCS Anti-Corruption Compliance Program??"
]

ground_truths = [
    # # 1. Definition of 'Client'
    # "Client: any person using the services of CPCS.",
    # # 2. Definition of 'CPCS Global Anti-Corruption Training'
    # "CPCS Global Anti-Corruption Training: the approved CPCS training course provided during the onboarding and induction process.",
    # # 3. Definition of 'Partners'
    # "Partners: includes any person for whom CPCS acts as a sub-contractor in any given project or person acting as sub-contractor to CPCS; and joint venture partner.",
    # # 4. Definition of 'Staff'
    "Staff: all CPCS employees (including those in management positions) and exclusive associates.",
    # 5. Definition of 'Due Diligence'
    "Due Diligence: submission for completion by the person being subject of the due diligence investigation CPCS Third Party Due Diligence Questionnaire (EN / FR).",
    # 6. Definition of 'Whistleblower'
    "Whistleblower: Any Staff who knows or suspects that another Staff or a member of the CPCS Board of Directors has committed or is about to commit an act contrary to the provisions of the Corruption of Foreign Public Officials Act and wishes to inform CPCS in a confidential manner."
]

# --- Run your RAG-based QA System ---
# Assume 'qa_chain' is your QA pipeline that you have already set up.
predictions = []
for question in test_questions:
    # Generate an answer from your RAG pipeline
    result = get_response(question)
    predictions.append(result)
    print(f"Question: {question}\nGenerated Answer: {result}\n{'-'*60}")

# --- Evaluate with Hugging Face's evaluate package (BERTScore) ---
bertscore_metric = evaluate.load("bertscore")
eval_results = bertscore_metric.compute(predictions=predictions, references=ground_truths, lang="en")

print("Hugging Face evaluate BERTScore results:")
print(eval_results)

# --- Evaluate with the standalone bert_score package ---
P, R, F1 = score(predictions, ground_truths, lang="en", verbose=True)
print("Standalone bert_score F1 (mean):", F1.mean().item())

# --- Tabulate the Results ---
results_data = []
for i, question in enumerate(test_questions):
    results_data.append({
        "Test Question": question,
        "Ground Truth": ground_truths[i],
        "Generated Answer": predictions[i],
        "Precision": eval_results["precision"][i],
        "Recall": eval_results["recall"][i],
        "F1 Score": eval_results["f1"][i]
    })

results_df = pd.DataFrame(results_data)

print("\nEvaluation Results Table:")
print(results_df)

# --- Display a nicely styled results table using CSS to hide the index ---
styled_df = (results_df.style
             .set_properties(**{'text-align': 'left', 'font-size': '14px'})
             .set_table_styles([
                 {'selector': '.row_heading, .blank', 'props': [('display', 'none')]},
                 {'selector': 'th', 'props': [('text-align', 'left'), ('font-size', '14px')]}
             ])
            )
display(styled_df)

This is the value of qa_chain from response: verbose=False combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:"), llm=OllamaLLM(model='llama3.2', base_url='http://10.50.10.240:10023/'), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_variable_name='context') retriever=VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x17c1438e0>, search_kwargs={'k': 3})
Question: What is the definition of 'Staff' accord

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Hugging Face evaluate BERTScore results:
{'precision': [0.8271013498306274, 0.8487792611122131, 0.934370219707489], 'recall': [0.8617295622825623, 0.8774533867835999, 0.9455167055130005], 'f1': [0.844060480594635, 0.862878143787384, 0.9399104714393616], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.50.0)'}


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.36 seconds, 8.41 sentences/sec
Standalone bert_score F1 (mean): 0.8822830319404602

Evaluation Results Table:
                                       Test Question  \
0  What is the definition of 'Staff' according to...   
1  What is the definition of 'Due Diligence' acco...   
2  What is the definition of 'Whistleblower' acco...   

                                        Ground Truth  \
0  Staff: all CPCS employees (including those in ...   
1  Due Diligence: submission for completion by th...   
2  Whistleblower: Any Staff who knows or suspects...   

                                    Generated Answer  Precision    Recall  \
0  I don't know the specific definition of "Staff...   0.827101  0.861730   
1  According to the provided context, "Due Dilige...   0.848779  0.877453   
2  According to the provided context, a "whistleb...   0.934370  0.945517   

   F1 Score  
0  0.844060  
1  0.862878  
2  0.939910  


,Test Question,Ground Truth,Generated Answer,Precision,Recall,F1 Score
0,What is the definition of 'Staff' according to CPCS Anti-Corruption Compliance Program??,Staff: all CPCS employees (including those in management positions) and exclusive associates.,"I don't know the specific definition of ""Staff"" according to CPCS Anti-Corruption Compliance Program, as the provided context only mentions certain groups or individuals that are required to complete training and adhere to the code of ethics (such as new staff, employees/EA's, and all staff), but does not explicitly define who constitutes ""staff"".",0.827101,0.861730,0.844060
1,What is the definition of 'Due Diligence' according to CPCS Anti-Corruption Compliance Program??,Due Diligence: submission for completion by the person being subject of the due diligence investigation CPCS Third Party Due Diligence Questionnaire (EN / FR).,"According to the provided context, ""Due Diligence"" refers to a process where the Divisional VP conducts research on a prospective Partner or Agent before entering into an agreement with them. It involves submitting completed CPCS Third Party Due Diligence Questionnaires to the Corporate Affairs Manager for review and approval.",0.848779,0.877453,0.862878
2,What is the definition of 'Whistleblower' according to CPCS Anti-Corruption Compliance Program??,Whistleblower: Any Staff who knows or suspects that another Staff or a member of the CPCS Board of Directors has committed or is about to commit an act contrary to the provisions of the Corruption of Foreign Public Officials Act and wishes to inform CPCS in a confidential manner.,"According to the provided context, a ""whistleblower"" for CPCS is defined as an employee (Staff) who knows or suspects that another Staff member or a member of the CPCS Board of Directors has committed or is about to commit an act contrary to the provisions of the Corruption of Foreign Public Officials Act.",0.934370,0.945517,0.939910
